# Classify name-unclassified funds via SEC Form ADV brochures (private machine)

Chain: fund -> adviser (`adv_advisers.csv`, built from ADV Schedule D 7.B.(1)) -> all Part 2A
brochures of that adviser -> keep only brochures that **name the fund** -> classify that brochure's
Item 8 text. No naming brochure = stays Unclassified. Brochure PDFs are saved to
`fund_classification/brochures/` as the citable evidence. Output: `adv_classification.csv`.

In [ ]:
import json, re, time
from pathlib import Path

import pandas as pd
import requests
from pypdf import PdfReader   # pip install pypdf

ROOT = Path.cwd() if (Path.cwd() / "hf_Valeri.xlsx").exists() else Path.cwd().parent
BRO = ROOT / "fund_classification" / "brochures"
BRO.mkdir(parents=True, exist_ok=True)
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/126.0 Safari/537.36",
      "Referer": "https://adviserinfo.sec.gov/"}

adv = pd.read_csv(ROOT / "adv_advisers.csv", dtype={"adviser_crd": str})
adv = adv[adv["adviser_crd"].notna()]
print(len(adv), "funds with an adviser,", adv["adviser_crd"].nunique(), "advisers")

## 0b. Complete missing advisers (only runs for rows without an adviser)

In [ ]:
full = pd.read_csv(ROOT / "adv_advisers.csv", dtype={"adviser_crd": str})
todo = full[full["adviser_crd"].isna()]
print(len(todo), "funds still without an adviser")

def norm(t):
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]", " ", str(t).upper())).strip()
SUF = {"LTD", "LIMITED", "LP", "L P", "LLC", "LDC", "INC", "THE"}
STOPW = {"THE", "MASTER", "FUND", "FUNDS", "LTD", "LIMITED", "LP", "LLC", "LDC", "INC", "SP", "SPC",
         "TRUST", "SERIES", "TRADING", "COMPANY", "PORTFOLIO", "PORTFOLIOS", "OFFSHORE", "CAYMAN",
         "INTERNATIONAL", "GLOBAL", "II", "III", "IV", "I"}
def name_keys(n):
    nn = norm(n)
    toks = nn.split()
    while toks and toks[-1] in SUF:
        toks = toks[:-1]
    return {nn} | ({" ".join(toks)} if len(toks) >= 3 else set())

ADV_TXT = ROOT / "fund_classification" / "adv_txt"
ADV_TXT.mkdir(parents=True, exist_ok=True)

def adv_text(crd):
    f = ADV_TXT / f"{crd}.txt"
    if f.exists():
        return f.read_text(errors="ignore")
    r = requests.get(f"https://reports.adviserinfo.sec.gov/reports/ADV/{crd}/PDF/{crd}.pdf",
                     headers=UA, timeout=120)
    time.sleep(1.5)
    if not r.ok or r.content[:4] != b"%PDF":
        f.write_text("")
        return ""
    tmp = ADV_TXT / f"{crd}.pdf"
    tmp.write_bytes(r.content)
    text = " ".join((pg.extract_text() or "") for pg in PdfReader(str(tmp)).pages)
    f.write_text(text)
    return text

for i, r in todo.iterrows():
    toks = [t for t in re.sub(r"[^A-Z0-9 ]", " ", str(r["name"]).upper()).split()
            if t not in STOPW and len(t) >= 3]
    if not toks:
        continue
    q = " ".join(toks[:2]) if len(toks) >= 2 else toks[0]
    s = requests.get("https://api.adviserinfo.sec.gov/search/firm",
                     params={"query": q, "hits": 8}, headers=UA, timeout=30)
    time.sleep(1)
    if not s.ok:
        continue
    ks = name_keys(r["name"])
    for h in s.json().get("hits", {}).get("hits", []):
        src = h["_source"]
        crd = str(src.get("firm_source_id"))
        if any(k in norm(adv_text(crd)) for k in ks):
            full.loc[i, ["adviser_name", "adviser_crd"]] = [src.get("firm_name"), crd]
            full.loc[i, "adviser_type"] = {"801": "RIA", "802": "ERA"}.get(
                str(src.get("firm_ia_full_sec_number", ""))[:3])
            print("matched:", r["name"], "->", src.get("firm_name"))
            break

full.to_csv(ROOT / "adv_advisers.csv", index=False)
adv = full[full["adviser_crd"].notna()]
print(len(adv), "funds with an adviser,", adv["adviser_crd"].nunique(), "advisers")

## 0c. Reverse search via curated manager names (evidence standard unchanged: fund must be listed in the adviser's 7.B.(1))

In [ ]:
cur = pd.read_csv(ROOT / "curated_managers.csv")
full = pd.read_csv(ROOT / "adv_advisers.csv", dtype={"adviser_crd": str})
cur_map = dict(zip(cur["fund"], cur["manager_guess"]))
todo = full[full["adviser_crd"].isna() & full["name"].isin(cur_map)]
print(len(todo), "funds to try via curated manager names")

for i, r in todo.iterrows():
    q = " ".join(str(cur_map[r["name"]]).split()[:3])
    s = requests.get("https://api.adviserinfo.sec.gov/search/firm",
                     params={"query": q, "hits": 8}, headers=UA, timeout=30)
    time.sleep(1)
    if not s.ok:
        continue
    ks = name_keys(r["name"])
    for h in s.json().get("hits", {}).get("hits", []):
        src = h["_source"]
        crd = str(src.get("firm_source_id"))
        if any(k in norm(adv_text(crd)) for k in ks):
            full.loc[i, ["adviser_name", "adviser_crd"]] = [src.get("firm_name"), crd]
            full.loc[i, "adviser_type"] = {"801": "RIA", "802": "ERA"}.get(
                str(src.get("firm_ia_full_sec_number", ""))[:3])
            print("matched:", r["name"], "->", src.get("firm_name"))
            break

full.to_csv(ROOT / "adv_advisers.csv", index=False)
adv = full[full["adviser_crd"].notna()]
print(len(adv), "funds with an adviser,", adv["adviser_crd"].nunique(), "advisers")

In [ ]:
def brochure_versions(crd):
    r = requests.get(f"https://api.adviserinfo.sec.gov/search/firm/{crd}", headers=UA, timeout=30)
    time.sleep(1)
    if not r.ok:
        print("  firm lookup failed", crd, r.status_code)
        return []
    src = r.json()["hits"]["hits"][0]["_source"]
    det = json.loads(src.get("iacontent") or src.get("content") or "{}").get("brochures", {}).get("brochuredetails", [])
    out = []
    for d in det if isinstance(det, list) else []:
        if isinstance(d, dict):
            vid = next((str(v) for k, v in d.items()
                        if ("vrsn" in k.lower() or "version" in k.lower()) and str(v).isdigit()), None)
            bname = next((str(v) for k, v in d.items() if "name" in k.lower()), "")
            if vid:
                out.append((vid, bname))
    return out

def fetch_brochure(crd, vid):
    dest = BRO / f"{crd}_{vid}.pdf"
    if dest.exists():
        return dest
    r = requests.get("https://files.adviserinfo.sec.gov/IAPD/Content/Common/"
                     f"crd_iapd_Brochure.aspx?BRCHR_VRSN_ID={vid}", headers=UA, timeout=60)
    time.sleep(1)
    if r.ok and r.content[:4] == b"%PDF":
        dest.write_bytes(r.content)
        return dest
    return None

pdfs = {}
for crd in sorted(adv["adviser_crd"].unique()):
    vers = brochure_versions(crd)
    pdfs[crd] = [p for vid, _ in vers if (p := fetch_brochure(crd, vid))]
    print(crd, "->", len(pdfs[crd]), "brochure(s)")

In [ ]:
RULES = [
    ("Fixed income / rates RV", ["FIXED INCOME", "FIRV", "RELATIVE VALUE", " RATES", "G-10",
        "GLOBAL RATES", "INFLATION", "BOND", "TERM CREDIT", "CONVEX", "TAIL RISK", "VOLATILITY"]),
    ("Global macro", ["MACRO", "ALL WEATHER", "PURE ALPHA", "OPTIMAL PORTFOLIO", "DMO"]),
    ("Credit", ["CREDIT", "ABS ", "HIGH YIELD", "DISTRESSED"]),
    ("Equity", ["EQUITY"]),
    ("Commodity", ["COMMODITY"]),
    ("Multi-strategy platform", ["MULTI-STRATEGY", "MULTI STRATEGY", "DIVERSIFIED ALPHA"]),
]

def classify(text):
    if not isinstance(text, str):
        return "Unclassified"
    t = " " + re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9\- ]", " ", text.upper())) + " "
    return next((lab for lab, kws in RULES if any(k in t for k in kws)), "Unclassified")

def norm(t):
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]", " ", str(t).upper())).strip()
SUF = {"LTD", "LIMITED", "LP", "L P", "LLC", "LDC", "INC", "THE"}
def name_keys(n):
    nn = norm(n)
    toks = nn.split()
    while toks and toks[-1] in SUF:
        toks = toks[:-1]
    return {nn} | ({" ".join(toks)} if len(toks) >= 3 else set())

def pdf_text(p):
    try:
        return " ".join((pg.extract_text() or "") for pg in PdfReader(str(p)).pages)
    except Exception:
        return ""

texts = {p: norm(pdf_text(p)) for ps in pdfs.values() for p in ps}

rows = []
for _, r in adv.iterrows():
    ks = name_keys(r["name"])
    naming = [p for p in pdfs.get(r["adviser_crd"], []) if any(k in texts[p] for k in ks)]
    strategies = {classify(texts[p]) for p in naming} - {"Unclassified"}
    rows.append({"lei": r["lei"], "name": r["name"], "adviser_name": r["adviser_name"],
                 "adviser_crd": r["adviser_crd"],
                 "brochure_evidence": "; ".join(p.name for p in naming),
                 "strategy_adv": strategies.pop() if len(strategies) == 1 else "Unclassified"})

res = pd.DataFrame(rows)
res.to_csv(ROOT / "adv_classification.csv", index=False)
print(res["strategy_adv"].value_counts().to_string())
res[res["strategy_adv"] != "Unclassified"][["name", "adviser_name", "strategy_adv", "brochure_evidence"]]